In [1]:
def generate_data():
	# ========== Generate simulated data ==========
	country = 'United_States'
	patterns = load_template_patterns(country, smooth=True, max_age=80)
	age_dist = load_age_distribution(country, max_age=80)
	
	# Generate the participants
	print(" Generating participants...")
	part_gen = ParticipantGenerator(age_dist['P'].values)
 
	# Calculate the number of participants per group
	n_part_grp = 250
	df_part = part_gen.generate([n_part_grp] * 2)
	print(f" Generated {len(df_part)} participants with 250 groups. \n")
	
	# Generate contact intensity matrices
	print(" Generating contact intensity matrices...")
	cint_mat_gen = ContactMatrixGenerator(patterns, age_dist['P'].values)
	cint_mat_list = [cint_mat_gen.generate(max_margin=20, seed=i) for i in range(2)]
 
	print(f" Generating contact records...")
	cnt_gen = ContactGenerator(df_part, cint_mat_list)
	df_cnt = cnt_gen.generate(seed=0)

	print(f" Generated {len(df_cnt)} contact records. \n")

	# Save the data
	print(" Saving simulation data...")
 
	# Prepare package
	data = {
		'participant': df_part,
		'contact': df_cnt,
		'age_dist': age_dist['P'].values,
		'age_dist_props': {
			'subgroup': np.ones((2, age_dist['P'].values.shape[0])) / 2
		},
		'cint_matrices': {
			'subgroup': {i: cint_mat_list[i] for i in range(len(cint_mat_list))}
		}
	}
 
	sim_data_path = 'HiBRC/fine_data.pkl'

	with open(sim_data_path, 'wb') as f:
		pickle.dump(data, f)
	
	print(f" Simulation data saved to {sim_data_path}\n")
 
	print(" Visualising simulation data...")
	# Plot the contact intensity matrices
	charts = [plot_mosaic(matrix, title=f'True Contact Intensity: Subgroup {i}') for i, matrix in enumerate(cint_mat_list)]
	chart = alt.hconcat(*charts)
	
	fig_path = 'HiBRC/fig/cint.png'
	chart.save(fig_path)
 
	# Plot marginal contact intensity
	fig_path = 'HiBRC/fig/cint_marginal.png'
	charts = [plot_mosaic_marginal(matrix.sum(axis=1), title=f'True Marginal Contact Intensity: Subgroup {i}') for i, matrix in enumerate(cint_mat_list)]
	chart = alt.hconcat(*charts)
	chart.save(fig_path)
 
	print(f" Figures saved to {fig_path}\n")

#generate_data()

In [2]:
import pickle

import math
import numpy as np
import pandas as pd
import os
import jax
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_value

# Get path to the root of this repository
from cntmosaic.datasets import load_template_patterns, load_age_distribution
from cntmosaic.sim import ParticipantGenerator, ContactMatrixGenerator, ContactGenerator
from cntmosaic.preprocess import make_train_data, make_full_grid
from cntmosaic.models import HiBRCfine
from cntmosaic.models.priors import PenalisedTensorSpline2D, HSGP2D
from cntmosaic.analysis import ModelSummariserSVI, ModelEvaluator, ModelVisualiser
from cntmosaic.vis import plot_mosaic, plot_mosaic_marginal, plot_mosaic_empirical
import altair as alt

In [3]:
with open('HiBRC/fine_data.pkl', 'rb') as f:
	data = pickle.load(f)

In [4]:
from cntmosaic.dataloader._dataloader import BaseLoader, CoordToColumns, DataLoader

In [5]:

# ===================================

# ========== Fit the model ==========
# Unpack the data    
df_part = data['participant']
df_cnt = data['contact']
age_dist = data['age_dist']
age_dist_props = data['age_dist_props']
cint_matrices = data['cint_matrices']

# Prepare training data
print(" Prepare training data...")
df_merged = pd.merge(df_cnt, df_part, on=['id', 'subgroup'], how='left')
df_train = make_train_data(df_merged, 'id', 'subgroup')
df_train['subgroup'] = pd.Categorical(df_train['subgroup'], categories=range(2))

# Compile the model
print(" Compiling the model...")
priors = {
	'rate': PenalisedTensorSpline2D(
	type='global',
	symmetric=True,
	grid_type='diff-age',
	neighborhood=8
),
	'subgroup': PenalisedTensorSpline2D(
	loc=age_dist_props['subgroup'],
	type='partial',
	event_dim=2,
	transform='ilr',
	neighborhood=8
)
}
sample = pd.read_csv('sample_prop.csv')

 Prepare training data...
 Compiling the model...


In [6]:
sample.sample(frac=1)

,pop_age,subgroup,prop
46,46,0,0.5
149,68,1,0.5
121,40,1,0.5
59,59,0,0.5
52,52,0,0.5
...,...,...,...
5,5,0,0.5
18,18,0,0.5
29,29,0,0.5
82,1,1,0.5


In [7]:
mapper = CoordToColumns(age_part='age_part', age_cnt='age_cnt',size_pop='size', age_pop='age_pop', grp_vars_part='subgroup', grp_vars_cnt='subgroup', )

/home/yiminglin/Downloads/cntmosaic/cntmosaic/dataloader/_dataloader.py:81: UserWarning: Variable of the same name exists in participants and contacts: {'subgroup'} 
Default behaviour will neglect variable in contacts, only keeping variable in participants
If wish to treat them as separate variable, please append suffixes to distinguish them
  warnings.warn(f'Variable of the same name exists in participants and contacts: {var} \n' +


In [8]:
loader = DataLoader(data['participant'], 
                    data['contact'], 
                    pd.DataFrame({'size': age_dist, 'age_pop':np.arange(0,81,dtype=int)}), 
                    mapper, 
                    [[sample.sample(frac=1), 'pop_age', 'subgroup', 'prop']])

In [9]:
ds = loader.load()

In [10]:
ds

<xarray.Dataset> Size: 533kB
Dimensions:            (index: 12069, age: 81, pop_prop_subgroup: 2)
Coordinates:
  * index              (index) int64 97kB 0 2 4 6 8 ... 13114 13116 13118 13120
  * age                (age) int64 648B 0 1 2 3 4 5 6 7 ... 74 75 76 77 78 79 80
    pop_prop_subgroup  (pop_prop_subgroup, age) float64 1kB 0.5 0.5 ... 0.5 0.5
Data variables:
    y                  (index) int64 97kB 0 0 0 0 1 0 0 1 0 ... 0 0 0 0 0 0 0 0
    log_N              (index) float32 48kB ...
    log_P              (age) float32 324B ...
    aid                (index) int64 97kB 0 0 0 0 0 0 0 ... 80 80 80 80 80 80 80
    bid                (index) int64 97kB 0 1 2 3 4 5 6 ... 74 75 76 77 78 79 80
    subgroup           (index) int64 97kB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
Attributes:
    grp_vars:  {'subgroup': [0, 1]}

In [11]:

model = HiBRCfine(ds,  priors)
model.print_model_shape()

# Fit the model
print(" Fitting the model...")
prng_key = jax.random.PRNGKey(0)
init_values = {'baseline': -model.log_P.mean()}

guide = AutoNormal(model.model, init_loc_fn=init_to_value(values=init_values))
model.run_inference_svi(prng_key, guide)

# Save the model
print(" Saving the model...")
model_path = 'HiBRC/model.pkl'

with open(model_path, 'wb') as f:
	pickle.dump(model, f)
print(f" Model saved to {model_path}\n")



            Trace Shapes:            
             Param Sites:            
            Sample Sites:            
            baseline dist       |    
                    value       |    
    rate/spline_coef dist       | 900
                    value       | 900
     subgroup/event plate     1 |    
subgroup/spline_coef dist     1 | 900
                    value     1 | 900
               data plate 12069 |    
            inv_disp dist 12069 |    
                    value 12069 |    
                 obs dist 12069 |    
                    value 12069 |    
 Fitting the model...


100%|██████████| 5000/5000 [01:07<00:00, 74.37it/s, init loss: 63911.8828, avg. loss [4751-5000]: 13328.2121]


 Saving the model...
 Model saved to HiBRC/model.pkl



In [12]:
with open('HiBRC/model.pkl', 'rb') as f:
	model = pickle.load(f)
# ========== Summarise the results ==========
print(" Summarising the results...")
summariser = ModelSummariserSVI(model)

print(" Evaluating the model...")
evaluator = ModelEvaluator(summariser, data['cint_matrices'])
df_cint_eval = evaluator.evaluate_cint()
df_mcint_eval = evaluator.evaluate_mcint()

df_cint_eval.to_csv(f'HiBRC/cint_eval_{500}_{2}.csv', index=False)
df_mcint_eval.to_csv(f'HiBRC/mcint_eval_{500}_{2}.csv', index=False)

print(" Visualising the results...")
visualiser = ModelVisualiser(summariser)

# Plot the contact intensity matrices
chart_dict = visualiser.plot_cint()
fig_path = 'HiBRC/fig/post_cint_matrix_500_2.png'
chart_dict['subgroup'].save(fig_path)

# Plot marginal contact intensity
chart_dict = visualiser.plot_mcint(evaluator)
fig_path = 'HiBRC/fig/post_cint_marginal_500_2.png'
chart_dict['subgroup'].save(fig_path)

 Summarising the results...
 Evaluating the model...
 Visualising the results...
